# Import Libraries

In [3]:
import gzip
import os
import polars as pl
import requests
import shutil

# Download the Data

In [4]:
TCGA_PROJECTS = [
    'TCGA-ACC', 'TCGA-BLCA', 'TCGA-BRCA', 'TCGA-CESC', 'TCGA-CHOL', 
    'TCGA-COAD', 'TCGA-DLBC', 'TCGA-ESCA', 'TCGA-GBM', 'TCGA-HNSC', 
    'TCGA-KICH', 'TCGA-KIRC', 'TCGA-KIRP', 'TCGA-LAML', 'TCGA-LGG', 
    'TCGA-LIHC', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-MESO', 'TCGA-OV', 
    'TCGA-PAAD', 'TCGA-PCPG', 'TCGA-PRAD', 'TCGA-READ', 'TCGA-SARC', 
    'TCGA-SKCM', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM', 
    'TCGA-UCEC', 'TCGA-UCS', 'TCGA-UVM'
]

In [ ]:
def download_tcga_data(project, type, mode=''):
    if type == "methylation":
        tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.methylation450.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/DNA_Methylation/{project}.gz'
    if type == "gene_expression":
        if mode == 'dea':   
            tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.star_counts.tsv.gz'
        else:
            tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.star_tpm.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/Gene_Expression/{project}.gz'
    if type == 'protein':
        tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.protein.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/Protein_Expression/{project}.gz'
    if type == 'clinical':
        tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.clinical.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/Clinical_Data/{project}.gz'

    decompressed_file_path = output_file_path.replace('.gz', '')
    if os.path.exists(decompressed_file_path):
        print(f"File already exists: {decompressed_file_path}")
        return
    
    response = requests.get(tcga_download_endpoint, stream=True)

    if response.status_code == 200:
        print(f"Starting the download for project {project}")
        os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
        with open(output_file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print(f"Downloaded file saved to {output_file_path}")
        
        with gzip.open(output_file_path, 'rb') as f_in:
            with open(output_file_path.replace('.gz', ''), 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        print(f"Decompressed file saved to {output_file_path.replace('.gz', '')}")

        os.remove(output_file_path)
        print(f"Removed compressed file: {output_file_path}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")

In [4]:
for project in TCGA_PROJECTS:
    download_tcga_data(project, "methylation")
    download_tcga_data(project, "gene_expression")
    download_tcga_data(project, "protein")

Starting the download for project TCGA-ACC
Downloaded file saved to UCSC_TCGA_Data/DNA_Methylation/TCGA-ACC.gz
Decompressed file saved to UCSC_TCGA_Data/DNA_Methylation/TCGA-ACC
Removed compressed file: UCSC_TCGA_Data/DNA_Methylation/TCGA-ACC.gz
Starting the download for project TCGA-ACC
Downloaded file saved to UCSC_TCGA_Data/Gene_Expression/TCGA-ACC.gz
Decompressed file saved to UCSC_TCGA_Data/Gene_Expression/TCGA-ACC
Removed compressed file: UCSC_TCGA_Data/Gene_Expression/TCGA-ACC.gz
Starting the download for project TCGA-ACC
Downloaded file saved to UCSC_TCGA_Data/Protein_Expression/TCGA-ACC.gz
Decompressed file saved to UCSC_TCGA_Data/Protein_Expression/TCGA-ACC
Removed compressed file: UCSC_TCGA_Data/Protein_Expression/TCGA-ACC.gz
Starting the download for project TCGA-BLCA
Downloaded file saved to UCSC_TCGA_Data/DNA_Methylation/TCGA-BLCA.gz
Decompressed file saved to UCSC_TCGA_Data/DNA_Methylation/TCGA-BLCA
Removed compressed file: UCSC_TCGA_Data/DNA_Methylation/TCGA-BLCA.gz
Sta

# Process the Data

## DNA Methylation

### Determine which probes to keep

In [ ]:
def create_probes_to_remove_set():
    hg38_genes_file_path = 'HM450.hg38.manifest.gencode.v36.tsv'  

    df = pl.read_csv(hg38_genes_file_path, separator='\t',
                     null_values=['NA'])

    selected_columns = df.select(['CpG_chrm', 'probeID', 'genesUniq', 'transcriptTypes'])

    no_gene_associated = set()
    sexual_chr_probes = set()
    chrM_probes = set()
    non_cpg_probes = set()
    non_standard_transcript_probes = set()

    standard_transcript_types = {"protein_coding", "lncRNA", "miRNA"}

    for row in selected_columns.iter_rows(named=True):
        chromosome = row['CpG_chrm']  
        probe_id = str(row['probeID']) 
        gene_id = str(row['genesUniq'])
        gene_names = gene_id.split(";") 
        transcript_types = row['transcriptTypes']
            
        if gene_id == 'None':
            no_gene_associated.add(probe_id)
            continue

        if not probe_id.startswith('cg'):
            non_cpg_probes.add(probe_id)
            continue

        if chromosome == 'chrY' or chromosome=='chrX':
            sexual_chr_probes.add(probe_id)
            continue

        if chromosome=='chrM':
            chrM_probes.add(probe_id)
            continue

        if transcript_types is not None:
            types = transcript_types.split(";")             
            zipped = list(zip(gene_names, types))

            if not any(t in standard_transcript_types for (_, t) in zipped):
                non_standard_transcript_probes.add(probe_id)
                continue

        if len(gene_names) == 1: 
            if '.' in gene_names[0]: # Remove AL/AF lncRNA
                non_standard_transcript_probes.add(probe_id)

    print(f"Number of probes without gene: {len(no_gene_associated)}")
    print(f"Number of non standard probes: {len(non_cpg_probes)}")
    print(f"Number of sexual chromosome probes: {len(sexual_chr_probes)}")
    print(f"Number of M chromosome probes: {len(chrM_probes)}")
    print(f"Number of non standard transcript type probes: {len(non_standard_transcript_probes)}")

    probes_to_remove = no_gene_associated.union(non_cpg_probes).union(sexual_chr_probes). \
        union(chrM_probes).union(non_standard_transcript_probes) 

    return probes_to_remove

In [6]:
probes_to_remove = create_probes_to_remove_set()

Number of probes without gene: 61891
Number of non standard probes: 2051
Number of sexual chromosome probes: 10619
Number of M chromosome probes: 6
Number of non standard transcript type probes: 25171


### Remove unwanted probes 

In [ ]:
def is_valid_probe(probe_name):
    return probe_name not in probes_to_remove

def process_project_methylation_files(project):
    print(f'Processing data for project {project}')
    project_file_path = f'UCSC_TCGA_Data/DNA_Methylation/{project}'
    output_file_path = ('UCSC_TCGA_Data_Clean/DNA_Methylation/'
        f'{project}_filtered_methylation.tsv')
    
    if os.path.exists(output_file_path):
        return

    df = pl.read_csv(project_file_path, separator='\t',
                     null_values=['NA'])
    print(f'Initial shape {df.shape}')
    df = df.rename({'Composite Element REF': 'Probe'}) 

    df_filtered = df.filter(~df['Probe'].is_in(probes_to_remove))

    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(df_filtered.shape)
    df_filtered.write_csv(output_file_path, separator='\t')
    print(f'Finished writing data to file {output_file_path}')

    # Remove the original file to save up space  
    #os.remove(project_file_path)
    #print(f"Deleted original file {project_file_path} to free up space.")

In [11]:
for project in TCGA_PROJECTS:
    process_project_methylation_files(project)

Processing data for project TCGA-ACC
Initial shape (486427, 81)
(386689, 81)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-ACC_filtered_methylation.tsv
Deleted raw file UCSC_TCGA_Data/DNA_Methylation/TCGA-ACC to free up space.
Processing data for project TCGA-BLCA
Initial shape (486427, 438)
(386689, 438)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-BLCA_filtered_methylation.tsv
Deleted raw file UCSC_TCGA_Data/DNA_Methylation/TCGA-BLCA to free up space.
Processing data for project TCGA-BRCA
Initial shape (486427, 894)
(386689, 894)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-BRCA_filtered_methylation.tsv
Deleted raw file UCSC_TCGA_Data/DNA_Methylation/TCGA-BRCA to free up space.
Processing data for project TCGA-CESC
Initial shape (486427, 313)
(386689, 313)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-CESC_filtered_methylation.tsv
Deleted raw file UCSC_TCGA_Data/DNA_Methylation/T

## Gene Expression and Copy Number Variation

### Determine which genes to keep

In [6]:
def create_gencode_df():
    human_gencode = 'gencode.v36.basic.annotation.gtf'

    human_gencode_df = pl.read_csv(human_gencode, separator='\t',
                        null_values=['NA'], has_header=False, 
                        comment_prefix='#', truncate_ragged_lines=True)

    human_gencode_df = human_gencode_df.rename({
        "column_1": "seqname",
        "column_2": "source",
        "column_3": "feature",
        "column_4": "start",
        "column_5": "end",
        "column_6": "score",
        "column_7": "strand",
        "column_8": "frame",
        "column_9": "attributes",
    })

    human_gencode_df = human_gencode_df.with_columns([
        human_gencode_df["attributes"].str.extract(r'gene_name "([^"]+)"', 1).alias("gene_name"),
        human_gencode_df["attributes"].str.extract(r'gene_type "([^"]+)"', 1).alias("gene_type"),
    ])

    human_gencode_df = human_gencode_df.select(["gene_name", "gene_type"]).unique()
    return human_gencode_df

human_gencode_df = create_gencode_df()

In [7]:
def create_genes_to_remove_set(human_gencode_df):
    hg38_genes_file_path = 'gencode.v36.annotation.gtf.gene.probemap' 

    df = pl.read_csv(hg38_genes_file_path, separator='\t',
                     null_values=['NA'])

    selected_columns = df.select(['id', 'gene', 'chrom'])

    sexual_chr_genes = set()
    chrM_probes = set()
    non_standard_genes = set()
    standard_transcript_types = {"protein_coding", "lncRNA", "miRNA"}

    gencode_map = dict(zip(human_gencode_df['gene_name'].to_list(),
                           human_gencode_df['gene_type'].to_list()))

    for row in selected_columns.iter_rows(named=True):
        chromosome = row['chrom']  
        gene_id = str(row['id']) 
        gene_name = str(row['gene'])

        if chromosome == 'chrY' or chromosome == 'chrX':
            sexual_chr_genes.add(gene_id)
            continue

        if chromosome=='chrM':
            chrM_probes.add(gene_id)
            continue

        if gene_name in gencode_map.keys():
            gene_type = gencode_map[gene_name]
            if gene_type not in standard_transcript_types:
                non_standard_genes.add(gene_id)
                continue
        
        if '.' in gene_name:
            non_standard_genes.add(gene_id) # Remove pseudogenes and AL/AF lncRNA

    print(f"Number of non standard genes: {len(non_standard_genes)}")
    print(f"Number of sexual chromosome probes: {len(sexual_chr_genes)}")
    print(f"Number of M chromosome probes: {len(chrM_probes)}")

    ensemblids_to_remove = non_standard_genes.union(sexual_chr_genes).union(chrM_probes)

    return ensemblids_to_remove

In [8]:
ensemblids_to_remove = create_genes_to_remove_set(human_gencode_df)

Number of non standard genes: 33041
Number of sexual chromosome probes: 2990
Number of M chromosome probes: 37


### Remove unwanted genes

In [1]:
def process_project_gene_level_files(project, type):
    print(f'Processing data for project {project}')
    if type == 'cnv':
        project_file_path = f'UCSC_TCGA_Data/Copy_Number_Variation/{project}'
        output_file_path = ('UCSC_TCGA_Data_Clean/Copy_Number_Variation/'
            f'{project}_filtered_cnv.tsv')
    elif type == 'gene_expression':
        project_file_path = f'UCSC_TCGA_Data/Gene_Expression/{project}'
        output_file_path = ('UCSC_TCGA_Data_Clean/Gene_Expression/'
            f'{project}_filtered_gene_expression.tsv')
    
    if os.path.exists(output_file_path):
        return

    df = pl.read_csv(project_file_path, separator='\t',
                     null_values=['NA'])
    print(f'Initial shape {df.shape}')
    df = df.rename({'Ensembl_ID': 'Gene'})  

    df_filtered = df.filter(~df['Gene'].is_in(ensemblids_to_remove))
    
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(df_filtered.shape)
    df_filtered.write_csv(output_file_path, separator='\t')
    print(f'Finished writing data to file {output_file_path}')

In [9]:
for project in TCGA_PROJECTS:
    process_project_gene_level_files(project, "gene_expression")

Processing data for project TCGA-ACC
Initial shape (60660, 80)
(24592, 80)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-ACC_filtered_gene_expression.tsv
Processing data for project TCGA-BLCA
Initial shape (60660, 429)
(24592, 429)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-BLCA_filtered_gene_expression.tsv
Processing data for project TCGA-BRCA
Initial shape (60660, 1227)
(24592, 1227)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-BRCA_filtered_gene_expression.tsv
Processing data for project TCGA-CESC
Initial shape (60660, 310)
(24592, 310)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-CESC_filtered_gene_expression.tsv
Processing data for project TCGA-CHOL
Initial shape (60660, 45)
(24592, 45)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-CHOL_filtered_gene_expression.tsv
Processing data for project TCGA-COAD
Initial shape (60660, 515)
(24592, 515)
Finish

## Protein Expression

### Determine which genes to keep

In [17]:
def create_protein_genes_to_remove_set(ensemblids_to_remove):
    protein_genes_file_path = 'TCGA_antibodies_descriptions.gencode.v36.tsv'  

    df = pl.read_csv(protein_genes_file_path, separator='\t',
                     null_values=['NA'])
    
    selected_columns = df.select(['peptide_target', 'validation_status', 'gene_id'])

    invalid_antibodies = set()
    ensembl_filtered = set()

    for row in selected_columns.iter_rows(named=True):
        peptide = str(row['peptide_target'])
        gene_id = str(row['gene_id'])
        status = row['validation_status']

        if status != "Valid":
            invalid_antibodies.add(peptide)
            continue
    
        if gene_id in ensemblids_to_remove:
            ensembl_filtered.add(peptide)
    
    print(f"Number of Invalid/Other antibodies: {len(invalid_antibodies)}")
    print(f"Number of Ensembl ID-removed peptides: {len(ensembl_filtered)}")

    peptide_to_remove = invalid_antibodies.union(ensembl_filtered)
    return peptide_to_remove

In [18]:
peptide_to_remove = create_protein_genes_to_remove_set(ensemblids_to_remove)

Number of Invalid/Other antibodies: 204
Number of Ensembl ID-removed peptides: 7


### Remove unwanted peptides

In [ ]:
# Doesnt have for LAML project
def process_project_protein_files(project):
    print(f'Processing data for project {project}')
    project_file_path = f'UCSC_TCGA_Data/Protein_Expression/{project}'

    if not os.path.exists(project_file_path):
        print(project_file_path)
        return

    output_file_path = ('UCSC_TCGA_Data_Clean/Protein_Expression/'
        f'{project}_filtered_pe.tsv')
    
    if os.path.exists(output_file_path):
        return

    df = pl.read_csv(project_file_path, separator='\t',
                     null_values=['NA'])
    print(f'Initial shape {df.shape}')

    df_filtered = df.filter(~df['peptide_target'].is_in(peptide_to_remove))
            
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(df_filtered.shape)
    df_filtered.write_csv(output_file_path, separator='\t')
    print(f'Finished writing data to file {output_file_path}')

In [20]:
for project in TCGA_PROJECTS:
    process_project_protein_files(project)

Processing data for project TCGA-ACC
Initial shape (487, 47)
(276, 47)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-ACC_filtered_pe.tsv
Processing data for project TCGA-BLCA
Initial shape (487, 344)
(276, 344)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-BLCA_filtered_pe.tsv
Processing data for project TCGA-BRCA
Initial shape (487, 920)
(276, 920)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-BRCA_filtered_pe.tsv
Processing data for project TCGA-CESC
Initial shape (487, 173)
(276, 173)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-CESC_filtered_pe.tsv
Processing data for project TCGA-CHOL
Initial shape (487, 31)
(276, 31)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-CHOL_filtered_pe.tsv
Processing data for project TCGA-COAD
Initial shape (487, 364)
(276, 364)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-COAD_fi

# Intersect the Samples Available for each Omic

In [ ]:
def intersect_project_samples(project):
    print(f'Beginning intersection for {project}')  
    if project == 'TCGA-LAML':
        print('Skipping LAML project for protein expression')
        return 0
    
    dna_methylation_path = f'UCSC_TCGA_Data_Clean/DNA_Methylation/{project}_filtered_methylation.tsv'
    gene_expression_path = f'UCSC_TCGA_Data_Clean/Gene_Expression/{project}_filtered_gene_expression.tsv'
    protein_expression_path = f'UCSC_TCGA_Data_Clean/Protein_Expression/{project}_filtered_pe.tsv'

    dna_df = pl.read_csv(dna_methylation_path, separator='\t')
    ge_df = pl.read_csv(gene_expression_path, separator='\t')
    pe_df = pl.read_csv(protein_expression_path, separator='\t')

    dna_sample_cols = dna_df.select(pl.col(pl.Float64)).columns 
    ge_sample_cols = ge_df.select(pl.col(pl.Float64)).columns  
    pe_sample_cols = pe_df.select(pl.col(pl.Float64)).columns
    
    first_col_dna = dna_df.columns[0]
    first_col_ge = ge_df.columns[0]
    first_col_pe = pe_df.columns[0]
    
    
    common_samples = list(set(dna_sample_cols) & set(ge_sample_cols) & set(pe_sample_cols))
    
    dna_cols_to_keep = [first_col_dna] + common_samples
    ge_cols_to_keep = [first_col_ge] + common_samples
    pe_cols_to_keep = [first_col_pe] + common_samples

    dna_df_filtered = dna_df.select(dna_cols_to_keep)
    ge_df_filtered = ge_df.select(ge_cols_to_keep)
    pe_df_filtered = pe_df.select(pe_cols_to_keep)
    print(f'dna methylation shape: {dna_df_filtered.shape}')
    print(f'gene expression shape: {ge_df_filtered.shape}')
    print(f'protein expression shape: {pe_df_filtered.shape}')

    dna_df_filtered.write_csv(dna_methylation_path, separator='\t')
    ge_df_filtered.write_csv(gene_expression_path, separator='\t')
    pe_df_filtered.write_csv(protein_expression_path, separator='\t')
    print(f'Finished intersecting matrices for project {project}')
    return dna_df_filtered.shape[1]

In [ ]:
sample_number = 0
for project in TCGA_PROJECTS:
    count = intersect_project_samples(project)
    sample_number += int(count)
print(f'Total samples: {sample_number}')

Beginning intersection for TCGA-ACC
dna methylation shape: (386689, 47)
gene expression shape: (24588, 47)
protein expression shape: (276, 47)
Finished intersecting matrices for project TCGA-ACC
Beginning intersection for TCGA-BLCA
dna methylation shape: (386689, 339)
gene expression shape: (24588, 339)
protein expression shape: (276, 339)
Finished intersecting matrices for project TCGA-BLCA
Beginning intersection for TCGA-BRCA
dna methylation shape: (386689, 656)
gene expression shape: (24588, 656)
protein expression shape: (276, 656)
Finished intersecting matrices for project TCGA-BRCA
Beginning intersection for TCGA-CESC
dna methylation shape: (386689, 171)
gene expression shape: (24588, 171)
protein expression shape: (276, 171)
Finished intersecting matrices for project TCGA-CESC
Beginning intersection for TCGA-CHOL
dna methylation shape: (386689, 30)
gene expression shape: (24588, 30)
protein expression shape: (276, 30)
Finished intersecting matrices for project TCGA-CHOL
Beginnin

# Verify Sample Type

### Remove samples that are not primary tumor

In [ ]:
def remove_non_primary_tumor_samples(project):
    print(f'Beginning verifying samples type for {project}')  
    if project == 'TCGA-LAML':
        print('Skipping LAML project for protein expression')
        return 0
    
    dna_methylation_path = f'UCSC_TCGA_Data_Clean/DNA_Methylation/{project}_filtered_methylation.tsv'
    gene_expression_path = f'UCSC_TCGA_Data_Clean/Gene_Expression/{project}_filtered_gene_expression.tsv'
    protein_expression_path = f'UCSC_TCGA_Data_Clean/Protein_Expression/{project}_filtered_pe.tsv'

    dna_df = pl.read_csv(dna_methylation_path, separator='\t')
    ge_df = pl.read_csv(gene_expression_path, separator='\t')
    pe_df = pl.read_csv(protein_expression_path, separator='\t')
    print(f'Initial number of samples {dna_df.shape[1]}')

    first_col_dna = dna_df.columns[0]
    first_col_ge = ge_df.columns[0]
    first_col_pe = pe_df.columns[0]

    project_sample_types = {}

    columns_to_keep = []  
    dna_sample_cols = dna_df.select(pl.col(pl.Float64)).columns 
    for column in dna_sample_cols:
        sample_type = column.split('-')[3][:2] 
        if sample_type not in project_sample_types:
            project_sample_types[sample_type] = 1
        else:
            project_sample_types[sample_type] += 1
        if sample_type == '01' or sample_type == '11':   
            columns_to_keep.append(column)

    print(project_sample_types)

    dna_cols_to_keep = [first_col_dna] + columns_to_keep
    ge_cols_to_keep = [first_col_ge] + columns_to_keep
    pe_cols_to_keep = [first_col_pe] + columns_to_keep
    
    dna_df_filtered = dna_df.select(dna_cols_to_keep)
    ge_df_filtered = ge_df.select(ge_cols_to_keep)
    pe_df_filtered = pe_df.select(pe_cols_to_keep)

    dna_df_filtered.write_csv(dna_methylation_path, separator='\t')
    ge_df_filtered.write_csv(gene_expression_path, separator='\t')
    pe_df_filtered.write_csv(protein_expression_path, separator='\t')
    print(f'Final number of samples {dna_df_filtered.shape[1]}')
    print(f'Finished verifying samples type for project {project}')
    return dna_df_filtered.shape[1]

In [ ]:
sample_number = 0
for project in TCGA_PROJECTS:
    count = remove_non_primary_tumor_samples(project)
    sample_number += count
print(f'Total samples: {sample_number}')

Beginning verifying samples type for TCGA-ACC
Initial number of samples 47
{'01': 46}
Final number of samples 47
Finished verifying samples type for project TCGA-ACC
Beginning verifying samples type for TCGA-BLCA
Initial number of samples 339
{'01': 338}
Final number of samples 339
Finished verifying samples type for project TCGA-BLCA
Beginning verifying samples type for TCGA-BRCA
Initial number of samples 656
{'01': 630, '11': 21, '06': 4}
Final number of samples 652
Finished verifying samples type for project TCGA-BRCA
Beginning verifying samples type for TCGA-CESC
Initial number of samples 171
{'01': 170}
Final number of samples 171
Finished verifying samples type for project TCGA-CESC
Beginning verifying samples type for TCGA-CHOL
Initial number of samples 30
{'01': 29}
Final number of samples 30
Finished verifying samples type for project TCGA-CHOL
Beginning verifying samples type for TCGA-COAD
Initial number of samples 250
{'01': 247, '02': 1, '06': 1}
Final number of samples 248

# Agreggate Dataframes by Omics

In [ ]:
def aggregate_dataframes_omics(type):
    if type == 'cnv':
        directory = 'UCSC_TCGA_Data_Clean/Copy_Number_Variation'
        id_col_name = 'Ensembl_ID'
    elif type == 'ge':
        directory = 'UCSC_TCGA_Data_Clean/Gene_Expression'
        id_col_name = 'Ensembl_ID'
    elif type == 'pe':
        directory = 'UCSC_TCGA_Data_Clean/Protein_Expression'
        id_col_name = 'AGID'
    elif type == 'dna':
        directory = 'UCSC_TCGA_Data_Clean/DNA_Methylation'
        id_col_name = 'Probe_ID'
    output_file_path = f'UCSC_TCGA_Data_Clean/Aggregated/{type}.tsv'
    all_gene_ids = set()

    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        df = pl.read_csv(file_path, separator='\t')
        gene_ids = df.select(df.columns[0]).to_series().to_list()
        all_gene_ids.update(gene_ids)

    aggregated_df = pl.DataFrame({id_col_name: list(all_gene_ids)})

    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        df = pl.read_csv(file_path, separator='\t')
        original_id_col = df.columns[0]
        df = df.rename({original_id_col: id_col_name})
        aggregated_df = aggregated_df.join(df, on=id_col_name, how='left', coalesce=True)

    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(aggregated_df.shape)
    aggregated_df.write_csv(output_file_path, separator='\t')
    print(f'Finished writing aggregated dataframe for {type} type.')

In [ ]:
#If working with protein expression, delete LAML project
ge_laml = 'UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-LAML_filtered_gene_expression.tsv'
dna_laml = 'UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-LAML_filtered_methylation.tsv'

if os.path.exists(ge_laml):
    os.remove(ge_laml)
    print(f'Removed file: {ge_laml}')
if os.path.exists(dna_laml):
    os.remove(dna_laml)
    print(f'Removed file: {dna_laml}')

In [ ]:
aggregate_dataframes_omics('ge')
aggregate_dataframes_omics('pe')
aggregate_dataframes_omics('dna')

# Remove Unneeded Files

In [19]:
def remove_files_in_directory(dir):
    for file in os.listdir(dir):
        file_path = os.path.join(dir, file)
        os.remove(file_path)

In [ ]:
## Free up device space if needed
directories = [
    #'UCSC_TCGA_Data/Copy_Number_Variation',
    #'UCSC_TCGA_Data/Gene_Expression',
    #'UCSC_TCGA_Data/DNA_Methylation',
    #'UCSC_TCGA_Data/Protein_Expression',
    #'UCSC_TCGA_Data_Clean/Copy_Number_Variation',
    #'UCSC_TCGA_Data_Clean/Gene_Expression',
    #'UCSC_TCGA_Data_Clean/DNA_Methylation',
    #'UCSC_TCGA_Data_Clean/Protein_Expression',
    #'UCSC_TCGA_Data_Clean/Aggregated',   # For BNs creation, need to keep the aggregated ge matrix
]

for dir in directories:
    remove_files_in_directory(dir)

# Normalization

In [ ]:
def min_max_scale(type):
    print(f'Beginning normalization for type {type}')

    if type == "gene_expression":
        file_path = 'UCSC_TCGA_Data_Clean/Aggregated/ge.tsv'
        output_file_path = 'UCSC_TCGA_Data_Clean/Aggregated/normalized_ge.tsv'
    elif type == "pe":
        file_path = 'UCSC_TCGA_Data_Clean/Aggregated/pe.tsv'
        output_file_path = 'UCSC_TCGA_Data_Clean/Aggregated/normalized_pe.tsv'

    df = pl.read_csv(file_path, separator='\t')
    print(f'Read file dataframe with shape {df.shape}')
    
    numeric_cols = df.select(pl.col(pl.Float64)).columns

    numeric_df = df.select(numeric_cols)

    min_vals = numeric_df.min_horizontal()
    max_vals = numeric_df.max_horizontal()

    normalized_df = (numeric_df - min_vals) / (max_vals - min_vals)
    result_df = df.with_columns(normalized_df)

    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(result_df.shape)
    result_df.write_csv(output_file_path, separator='\t')
    print(f'Finished writing data to file {file_path}')

In [ ]:
min_max_scale('pe')
min_max_scale('gene_expression')

# Replace NaN for null

In [41]:
def replace_all_nan():
    files_path = ['UCSC_TCGA_Data_Clean/Aggregated/normalized_ge.tsv', 'UCSC_TCGA_Data_Clean/Aggregated/normalized_pe.tsv', \
                 'UCSC_TCGA_Data_Clean/Aggregated/dna.tsv', 'UCSC_TCGA_Data_Clean/Aggregated/ge.tsv']
    for file_path in files_path:
        if not os.path.exists(file_path):
            continue
        df = pl.read_csv(file_path, separator='\t')
        print((df.null_count().sum_horizontal() > 0).any())
        replaced_nan_df = df.fill_nan(None)
        print((replaced_nan_df.null_count().sum_horizontal() > 0).any())
        replaced_nan_df.write_csv(file_path, separator='\t')

In [ ]:
replace_all_nan()

# Remove Missing Values

In [ ]:
def remove_missing_values_threshold(df, var): 
    numeric_cols = df.columns[1:] 
    num_columns = len(numeric_cols)
    na_threshold = num_columns * 0.05
    variance_threshold = var

    if var != 0:
        filtered_df = df.with_columns(
                pl.sum_horizontal([pl.col(c).is_null() for c in numeric_cols])
                    .alias("na_count"),
                pl.concat_list([pl.col(c).cast(pl.Float64) for c in numeric_cols])
                .list.var()
                .alias("row_var")
            )
        
        filtered_df = filtered_df.filter(
                (pl.col("na_count") <= na_threshold) &
                (pl.col("row_var") >= variance_threshold) 
            ).drop(["na_count", "row_var"]) 
    else: 
        filtered_df = df.with_columns(
                pl.sum_horizontal([pl.col(c).is_null() for c in numeric_cols])
                    .alias("na_count")
            )
        
        filtered_df = filtered_df.filter(
                (pl.col("na_count") <= na_threshold)
            ).drop(["na_count"])

    print(filtered_df.shape)
    return filtered_df

In [ ]:
def remove_missing_values_methylation():
    directory = 'UCSC_TCGA_Data_Clean/Aggregated'
    output_directory = 'UCSC_TCGA_Data_Clean/Clean/'
    for file in os.listdir(directory):
        if file.startswith('dna'):
            output_file_path = output_directory + file.split('.')[0] + '_clean.tsv'            
            file_path = os.path.join(directory, file)
            df = pl.read_csv(file_path, separator='\t', null_values=['NA'])
            print(f'Read file {file} dataframe with shape {df.shape}')

            filtered_df = remove_missing_values_threshold(df, 0.05)

            os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
            print(filtered_df.shape)
            filtered_df.write_csv(output_file_path, separator='\t')
            print(f'Finished writing data to file {output_file_path}')

In [77]:
remove_missing_values_methylation()

Read file dna.tsv dataframe with shape (386689, 6045)
(29686, 6045)
(29686, 6045)
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/dna_clean.tsv


In [ ]:
def remove_missing_and_zero_expression():
    directory = 'UCSC_TCGA_Data_Clean/Aggregated'
    output_directory = 'UCSC_TCGA_Data_Clean/Clean/'
    for file in os.listdir(directory):
        if file.startswith('normalized'):
            output_file_path = output_directory + file.split('.')[0].split('_')[1] + '_clean.tsv'
            file_path = os.path.join(directory, file)
            df = pl.read_csv(file_path, separator='\t', null_values=['NA'])
            print(f'Read file {file} dataframe with shape {df.shape}')

            if not file.startswith('normalized_pe'):
                filtered_df = remove_missing_values_threshold(df, 0)
            else:
                filtered_df = df #redundant

            os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
            print(filtered_df.shape)
            filtered_df.write_csv(output_file_path, separator='\t')
            print(f'Finished writing data to file {output_file_path}')

In [ ]:
remove_missing_and_zero_expression()

# Imputation

In [ ]:
### Used the median to account for skewed data
def impute_row_with_median(df):
    numeric_cols = df.columns[1:] 

    row_medians = (
        df.select(
            pl.concat_list(numeric_cols)  
              .list.drop_nulls()          
              .list.median()              
              .alias("row_median")
        )
        ["row_median"]                   
    )

    df = df.with_columns(
        [
            pl.when(pl.col(c).is_null())
              .then(row_medians)          
              .otherwise(pl.col(c))
              .alias(c)
            for c in numeric_cols
        ]
    )

    return df

def perform_imputation(type):
    print(f'Processing data for type {type}')
    if type == "ge":
        file_path = 'UCSC_TCGA_Data_Clean/Clean/ge_clean.tsv'
    elif type == "pe":
        file_path = 'UCSC_TCGA_Data_Clean/Clean/pe_clean.tsv'
    elif type == 'dna':
        file_path = 'UCSC_TCGA_Data_Clean/Clean/dna_clean.tsv' 

    df = pl.read_csv(file_path, separator='\t')
    resulting_df = impute_row_with_median(df)

    resulting_df.write_csv(file_path, separator='\t')
    print(f'Finished writing data to file {file_path}')

In [ ]:
perform_imputation("dna")
perform_imputation("ge")
perform_imputation("pe")

Processing data for type ge
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/ge_clean.tsv
Processing data for type pe
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/pe_clean.tsv


## Create Clinical Data

In [72]:
for project in TCGA_PROJECTS:
    download_tcga_data(project, "clinical")

Starting the download for project TCGA-ACC
Downloaded file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-ACC.gz
Decompressed file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-ACC
Removed compressed file: UCSC_TCGA_Data/Clinical_Data/TCGA-ACC.gz
Starting the download for project TCGA-BLCA
Downloaded file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-BLCA.gz
Decompressed file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-BLCA
Removed compressed file: UCSC_TCGA_Data/Clinical_Data/TCGA-BLCA.gz
Starting the download for project TCGA-BRCA
Downloaded file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-BRCA.gz
Decompressed file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-BRCA
Removed compressed file: UCSC_TCGA_Data/Clinical_Data/TCGA-BRCA.gz
Starting the download for project TCGA-CESC
Downloaded file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-CESC.gz
Decompressed file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-CESC
Removed compressed file: UCSC_TCGA_Data/Clinical_Data/TCGA-CESC.gz
Starting the download for pr

In [10]:
def create_clinical_data_df():
    directory = 'UCSC_TCGA_Data/Clinical_Data/'
    output_file_path = 'UCSC_TCGA_Data_Clean/Clean/clinical.tsv'
    barcode_project_list = []
    if len(os.listdir('UCSC_TCGA_Data_Clean/Clean/')) > 0:
        pe_clean = pl.read_csv('UCSC_TCGA_Data_Clean/Clean/pe_clean.tsv', separator='\t')
        tcga_barcodes = set(pe_clean.columns[1:])
    
    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        df = pl.read_csv(file_path, separator='\t', schema_overrides={'submitter_id.annotations': pl.Utf8()},ignore_errors=True)

        selected_columns = df.select(['sample', 'project_id.project', 'tissue_type.samples'])

        for row in selected_columns.iter_rows(named=True):
            sample_code = str(row['sample'])
            project_id = str(row['project_id.project']).split('-')[1]
            sample_type = str(row['tissue_type.samples'])
            if sample_type == 'Normal':
                project_id = 'NORMAL'

            if sample_code not in tcga_barcodes:
                continue
            barcode_project_list.append((sample_code, project_id))

    barcode_project_df = pl.DataFrame(barcode_project_list, schema=["Barcode", "Tumor"], orient='row')

    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    barcode_project_df.write_csv(output_file_path, separator='\t')
    return barcode_project_df.shape

In [11]:
df_shape = create_clinical_data_df()
print(df_shape)

(6044, 2)
